# Proyecto Telecom, ultima parte del proyecto final 

### Adjunto link de presentacion PDF https://drive.google.com/drive/folders/1Gy4N1M8nZ1vflcW3Chz-ryDB7kFMbQ2d?usp=drive_link
# Encuentra operadores menos eficaces 
## Se considera un operador ineficaz al que tiene un agran cantidad de llamadas entrantes perdidas y un tiempo de espera prolongado para las llamadas entrantes, ademnas de no realizar llamadas salientes.

# PASO 1: Carga de las librerias Python y primer acercamiento a los datos

In [ ]:
# Librerias para el analisis 
import pandas as pd 
import datetime as dt 
import numpy as np 
import scipy.stats as stats 
import matplotlib.pyplot as plt 

In [ ]:
# Carga del DataSet 
telecom_operador  = pd.read_csv('telecom_dataset_new.csv')

In [ ]:
# Carga del DataSet 
telecom_clients = pd.read_csv('telecom_clients.csv')

In [ ]:
telecom_operador.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  object 
 2   direction            53902 non-null  object 
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(3)
memory usage: 3.3+ MB


In [ ]:
telecom_operador.head(5)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25


In [ ]:
telecom_clients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      732 non-null    int64 
 1   tariff_plan  732 non-null    object
 2   date_start   732 non-null    object
dtypes: int64(1), object(2)
memory usage: 17.3+ KB


In [ ]:
# Primera vista de los datos 
telecom_clients.head(5)

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


### En el primer acercamiento a los datos, tenemos columnas con valores incorrectos, ademas de valores NaN, en una de las columnas importantes para el nalisis

In [ ]:
# Busqueda de duplicados en el DataFrame 
telecom_operador.duplicated().sum()

4900

### Me parece razonble encontrar duplicados en una muestra de este tipo, varias llamadas pequeñas en un mismo dia, realizadas por el mismo operador,y concidencias en tiempo me resulta algo que se puede considerar normal. Es u porcentaje pequeño a comparacion de la muestra aprximadamente un 9%, es prudente dejar la muestra tal cual, y conforme avance el analisis teomar la decision de eliminar o concerbar para efecto del resultado. 

In [ ]:
# Busquedaq de valores Null en el DataFrame
for columna in telecom_operador.columns:
    nulos = telecom_operador[columna].isnull().sum()
    if nulos > 0:
        print(f"La columna '{columna}' tiene {nulos} valores Null ")

La columna 'internal' tiene 117 valores Null 
La columna 'operator_id' tiene 8172 valores Null 


In [ ]:
# Filtrado de valores NaN 
telecom_operador[telecom_operador['operator_id'].isnull()][['direction', 'is_missed_call', 'internal']]

,direction,is_missed_call,internal
0,in,True,False
7,in,True,False
9,in,True,False
17,in,True,False
27,in,True,False
...,...,...,...
53876,in,True,False
53880,in,True,False
53889,in,True,False
53893,in,True,False


In [ ]:
telecom_operador[telecom_operador['operator_id'].isnull()]['direction'].unique()

array(['in', 'out'], dtype=object)

In [ ]:
telecom_operador[telecom_operador['operator_id'].isnull()]['is_missed_call'].unique()

array([ True, False])

In [ ]:
telecom_operador[telecom_operador['operator_id'].isnull()]['internal'].unique()

array([False, True, nan], dtype=object)

### Busqueda de duplicados y nulos en telecom_clientes

In [ ]:
telecom_clients.duplicated().sum()

0

In [ ]:
telecom_clients.isnull().sum()

user_id        0
tariff_plan    0
date_start     0
dtype: int64

### Se conservaron los datos duplicados y se remplazaron los valores NaN con 0 , de esta modo, al realizar operaciones o manipulacion que involucre numeros la muestra no se vera afectada.

# PASO 2: Conversion de la columnas a su formato correcto 

In [ ]:
telecom_operador['date'] = pd.to_datetime(telecom_operador['date'])

telecom_operador.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype                                
---  ------               --------------  -----                                
 0   user_id              53902 non-null  int64                                
 1   date                 53902 non-null  datetime64[ns, pytz.FixedOffset(180)]
 2   direction            53902 non-null  object                               
 3   internal             53785 non-null  object                               
 4   operator_id          45730 non-null  float64                              
 5   is_missed_call       53902 non-null  bool                                 
 6   calls_count          53902 non-null  int64                                
 7   call_duration        53902 non-null  int64                                
 8   total_call_duration  53902 non-null  int64                                
dtypes: bool

In [ ]:
telecom_clients['date_start'] = pd.to_datetime(telecom_clients['date_start'])

telecom_clients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      732 non-null    int64         
 1   tariff_plan  732 non-null    object        
 2   date_start   732 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 17.3+ KB


## Aquí vale la pena notar una diferencia interesante entre las dos columnas:
## telecom_operador['date'] → quedó como datetime64[ns, pytz.FixedOffset(180)] (con zona horaria)
## telecom_clients['date_start'] → quedó como datetime64[ns] (sin zona horaria)
## Esto tiene sentido si date_start en el archivo original no traía esa información de zona horaria (+03:00) en el texto, así que pandas no tuvo nada que interpretar ahí.
## ¿Esto es un problema? Depende de si en algún momento necesitamos comparar o combinar fechas de ambas tablas directamente (por ejemplo, restar una fecha de otra). Si intentamos operar entre una columna con zona horaria y otra sin ella, pandas normalmente lanza un error.
## Por ahora, dado que aún no sabemos si vamos a necesitar cruzar esas dos columnas de fecha directamente, se continuara con la limpieza de los datos 

# PASO 3: Busqueda de estadisticas descriptibas generales de las columnas numericas.

In [ ]:
telecom_operador.describe().round(2)

,user_id,operator_id,calls_count,call_duration,total_call_duration
count,53902.00,45730.00,53902.00,53902.00,53902.00
mean,167295.34,916535.99,16.45,866.68,1157.13
std,598.88,21254.12,62.92,3731.79,4403.47
min,166377.00,879896.00,1.00,0.00,0.00
25%,166782.00,900788.00,1.00,0.00,47.00
50%,167162.00,913938.00,4.00,38.00,210.00
75%,167819.00,937708.00,12.00,572.00,902.00
max,168606.00,973286.00,4817.00,144395.00,166155.00


### Al ver que en calls_count el percentil 75% es 12 pero el máximo es 4817, ¿qué sugiere esa diferencia tan grande entre "donde está la mayoría de los datos" (percentil 75) y el valor máximo?
### Un pequeño grupo de operadores está haciendo muchísimas más llamadas (o con duraciones mucho más largas) que el resto, mientras que el 75% de los registros está muy por debajo de eso.Ese patrón, donde hay pocos valores extremos muy alejados del resto, se llama comúnmente tener outliers (valores atípicos) o una distribución con cola larga (sesgada hacia la derecha). 

In [ ]:
# Calculando el percentil 75 sobre calls_count
umbral_p75 = telecom_operador['calls_count'].quantile(0.75)

In [ ]:
# Filtrado de user_id, que cumplen este percentil 'umbral_p75'
operador_alto = telecom_operador[telecom_operador['calls_count'] > umbral_p75]

In [ ]:
operador_alto['operator_id'].unique()

array([880028., 880026.,     nan, 882686., 882684., 902510., 902532.,
       882690., 882688., 939476., 939474., 939470., 879898., 879896.,
       880240., 888534., 887282., 887276., 907174., 906866., 959312.,
       883940., 883942., 884412., 906680., 891410., 891414., 891416.,
       884946., 893674., 892028., 923250., 884478., 925826., 925828.,
       908958., 908960., 885890., 885876., 925922., 893420., 893402.,
       887992., 892798., 890404., 890416., 890412., 890410., 890402.,
       890406., 890422., 891154., 890420., 891160., 891162., 891170.,
       891152., 890618., 900892., 916596., 917234., 932502., 932500.,
       932050., 932498., 943050., 941628., 954642., 960296., 960294.,
       972410., 972412., 889410., 899082., 891252., 891988., 889754.,
       893804., 900180., 899250., 892538., 900594., 903254., 902720.,
       913868., 895774., 895776., 894656., 896016., 906396., 906404.,
       906406., 906408., 906400., 906394., 906410., 906412., 896428.,
       954750., 9105

### Con este filtro se obtienen los usuarios con una taza alta de llamadas, se quedara en este punto para retomarlo mas adelante para ejecutar comparitbas u obtener mas informacion mediante cruces o comparaciones 

### Busqueda de tasa de llamadas perdidas por usuario 

In [ ]:
telecom_operador.columns

Index(['user_id', 'date', 'direction', 'internal', 'operator_id',
       'is_missed_call', 'calls_count', 'call_duration',
       'total_call_duration'],
      dtype='object')

In [ ]:
# Llmadas entrantes
llamadas_entrantes = telecom_operador[telecom_operador['direction'] == 'in'].copy()
llamadas_entrantes['direction'].count()

21985

### Busqueda de tiempo de espera en llamadas 

In [ ]:
llamadas_entrantes['tiempo_espera'] = llamadas_entrantes ['total_call_duration']-llamadas_entrantes[ 'call_duration']
llamadas_entrantes.head(5)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration,tiempo_espera
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4,4
7,166377,2019-08-05 00:00:00+03:00,in,False,NaN,True,6,0,35,35
9,166377,2019-08-06 00:00:00+03:00,in,False,NaN,True,4,0,62,62
17,166377,2019-08-07 00:00:00+03:00,in,False,NaN,True,2,0,24,24
26,166377,2019-08-12 00:00:00+03:00,in,False,880028.0,False,1,407,411,4


In [ ]:
metricas_operador = llamadas_entrantes.groupby('operator_id').agg(total_llamadas_in = ('is_missed_call','count'), llamadas_perdidas = ('is_missed_call', 'sum'), 
                                                                  promedio_espera = ('tiempo_espera', 'mean'))
metricas_operador.head(5).round(2)

,total_llamadas_in,llamadas_perdidas,promedio_espera
operator_id,,,
879896.0,23,0,26.91
879898.0,64,0,28.20
880020.0,8,0,7.25
880022.0,8,0,14.00
880026.0,19,0,7.89


In [ ]:
metricas_operador['llamadas_perdidas'].unique()

array([ 0,  3,  4,  5,  2, 21, 22,  1, 11, 10,  6,  7,  8, 14, 29,  9, 15,
       12])

### Con esta agrupacion, obtenemos el numero de llamadas entrantes, cuantas llamadas perdidas tenemos y tambien el promedio de espera de las llamadas, que obtenemos, con la resta de las columnas total_call_duratio - call_duration 

### Busqueda de llamadas salientes 

In [ ]:
# Llamadas salientes 
llamadas_salientes = telecom_operador[telecom_operador['direction'] == 'out'].copy()
llamadas_salientes['direction'].count()

31917

In [ ]:
metricas_operador_out = llamadas_salientes.groupby('operator_id').agg(total_llamadas_salientes = ('calls_count', 'sum'))
metricas_operador_out.head(5)

,total_llamadas_salientes
operator_id,
879896.0,1071
879898.0,7856
880020.0,46
880022.0,211
880026.0,2414


In [ ]:
# Merge() para tener la informacion en una solo tabla 
total_metricas_op = pd.merge(metricas_operador, metricas_operador_out, on='operator_id', how='outer')
total_metricas_op.head(5)

,total_llamadas_in,llamadas_perdidas,promedio_espera,total_llamadas_salientes
operator_id,,,,
879896.0,23.0,0.0,26.913043,1071.0
879898.0,64.0,0.0,28.203125,7856.0
880020.0,8.0,0.0,7.250000,46.0
880022.0,8.0,0.0,14.000000,211.0
880026.0,19.0,0.0,7.894737,2414.0


### Un operador que solo tiene llamadas entrantes esta en la izquierda, y uno con solo salientes esta en la derecha, tenemos 338 operadores que solo hacen llamadas, no reciben ninguna, 210 operadores que solo reciben llamadas, no hacen ninguna, estos dos datos dan como resultado 548 operadores, pero 544 son capaces de realizar y recibir llamadas dentro de su jornada de trabajo

In [ ]:
total_metricas_op.isnull().sum()

total_llamadas_in           338
llamadas_perdidas           338
promedio_espera             338
total_llamadas_salientes    210
dtype: int64

In [ ]:
total_metricas_op.fillna(0, inplace=True)

In [ ]:
total_metricas_op.isnull().sum()

total_llamadas_in           0
llamadas_perdidas           0
promedio_espera             0
total_llamadas_salientes    0
dtype: int64

In [ ]:
total_metricas_op.columns

Index(['total_llamadas_in', 'llamadas_perdidas', 'promedio_espera',
       'total_llamadas_salientes'],
      dtype='object')

In [ ]:
total_metricas_op.reset_index(inplace = True)

### Ahora con los datos limpios y las metricas solicitadas es hora de comenzar con la evaluacion de parametros para determinar si un operador ineficaz o no

# Paso 4: Establecer criterios para la evaluación de los operadores 

In [ ]:
total_metricas_op.describe().round(2)

,operator_id,total_llamadas_in,llamadas_perdidas,promedio_espera,total_llamadas_salientes
count,1092.00,1092.00,1092.00,1092.00,1092.00
mean,925517.54,12.83,0.68,39.65,612.95
std,22854.53,19.96,2.24,82.35,3103.14
min,879896.00,0.00,0.00,0.00,0.00
25%,906393.50,0.00,0.00,0.00,2.00
50%,925105.00,3.00,0.00,18.98,36.50
75%,944211.50,17.00,0.00,43.00,388.75
max,973286.00,121.00,29.00,1039.54,64897.00


In [ ]:
# Definicion de los umbrales para encontrar los parametros del criterio de evaluacion 
umbral_90_llp = total_metricas_op['llamadas_perdidas'].quantile(0.90)
umbral_90_pe = total_metricas_op['promedio_espera'].quantile(0.90)
umbral_10_tlls = total_metricas_op['total_llamadas_salientes'].quantile(0.10)

print(umbral_90_llp)
print(umbral_90_pe)
print(umbral_10_tlls)

2.0
84.2076923076923
0.0


In [ ]:
# Filtrado con las tres condiciones para los criterios 
operador_ineficiente = total_metricas_op[(total_metricas_op['llamadas_perdidas'] > umbral_90_llp) | (total_metricas_op['promedio_espera'] > umbral_90_pe) | (total_metricas_op['total_llamadas_salientes'] <= umbral_10_tlls)]

In [ ]:
operador_ineficiente.shape

(337, 5)

In [ ]:
total_metricas_op['llamadas_perdidas'].count()

1092

In [ ]:
operador_ineficiente['promedio_espera'].mean().round(2)

90.77

In [ ]:
total_metricas_op['total_llamadas_salientes'].sum()

669343.0

In [ ]:
# Filtrado de cada condicion por separado 
total_metricas_op[total_metricas_op['llamadas_perdidas'] > umbral_90_llp].shape

(79, 5)

In [ ]:
total_metricas_op[total_metricas_op['promedio_espera'] > umbral_90_pe].shape

(110, 5)

In [ ]:
total_metricas_op[total_metricas_op['total_llamadas_salientes'] <= umbral_10_tlls].shape 

(210, 5)

In [ ]:
len(operador_ineficiente) / len(total_metricas_op) * 100

30.86080586080586

In [ ]:
# Filtro para operadores eficientes, aplicando los mismos criterios 
operador_eficiente = total_metricas_op[~((total_metricas_op['llamadas_perdidas'] > umbral_90_llp) | (total_metricas_op['promedio_espera'] > umbral_90_pe) | (total_metricas_op['total_llamadas_salientes'] <= umbral_10_tlls))]

In [ ]:
operador_eficiente.shape

(755, 5)

In [ ]:
operador_eficiente['promedio_espera'].mean().round(2)

16.83

In [ ]:
len(operador_eficiente) / len(total_metricas_op) * 100

69.13919413919413

### Los tres criterios que se definieron y con los que se construyo el filtro fueron:

### Muchas llamadas entrantes perdidas.
### Tiempo de espera prolongado.
### Pocas llamadas salientes realizadas.

### En esta parte del analisis exploratorio nos encontramos con un filtro que se desarrollo de varias maneras diferentes, ya que no se podia obtener un resultado consistente, el objetivo del analisis es encontrar a los operadores que cumplan con almenos una de las especificaciones pra ser considerado un operador ineficiente, se abordo el filtro con el operador logico &, el cual arrojo un resultado en 0 lo cual se puede deducir que no es lo normal dentro de un análisisde este tipo, ya que este filtro pedía cumplir con las 3 condiciones de evaluación, lo cual era difícil, al filtrar percentil por percentil en los parametros, arrojaba resultado en las tres condiciones, pero se debe evaluar con el filtro definido, al final se opto por usar el operador logico '|', el cual nos dio resultados consistentes en la evaluación arrojando un 30.86% de operadores ineficientes.

# Paso 5 Prueba estadistica.

In [ ]:
#Prueba estadistica con ttest_ind()

diferencia_espera = stats.ttest_ind(operador_eficiente['promedio_espera'], operador_ineficiente['promedio_espera'])
print(diferencia_espera)


alpha = 0.05

if diferencia_espera.pvalue < alpha:
    print(f"Rechazamos H0, Si existe diferencia entre los tiempos de espera entre operadores, pvalue{diferencia_espera.pvalue:.4f} < alpha={alpha}")
else:
    print(f"NO Rechazamos HO, No existe diferencia significativa entre los grupos de operadores, pvalue{diferencia_espera.pvalue:.4f} < alpha={alpha}")



Ttest_indResult(statistic=-15.057284073765096, pvalue=1.0875085761524253e-46)
Rechazamos H0, Si existe diferencia entre los tiempos de espera entre operadores, pvalue0.0000 < alpha=0.05


# Concluciones 

### De una muestra de 1092 usuarios, llamada total_metricas_operador, se obtuvo 337 operadores ineficientes y 755 operadores eficientes, en los requerimientos, se pide evaluar al menos uno de los criterios, dentro de los estándares de satisfacción de un call center, el tiempo de espera en las llamadas es una métrica de vital importancia, por esa razón se tomó como criterio para hacer las comparaciones dentro del análisis
### Dentro de nuestra prueba ttest() el valor pvalue nos da un resultado de 1.0875085761524253e-46, el cual es menor que nuestro alpha de 0.05, por lo cual rechazamos H0, existe diferencia significativa entre los grupos de operadores.
### Como sugerencia para la supervisión, se puede implementar un esquema de metas que escale de menor a mayor para el grupo de operadores ineficientes, pequeños pasos que se midan por día, después semana y por último al mes. Dejando en claro que los resultados serán tomados en cuenta para evaluación interna, y posible renovación de contrato con la empresa prestadora de servicio.

### Muchas gracias por su tiempo, y por revisar mi proyecto, quedo atento para cualquier correccion, un gran saludo.

### ATTE. Cesar Rodriguez Jr. Data Analist 